# 02 · Spark + Sedona: exact Kerala count, cleaning, Spark vs Sedona speed test

**Run this in Google Colab.**

**Before you start:** Runtime → Change runtime type → **CPU**, and turn on **High-RAM** if you see it. Do not pick a GPU.

What it does:
1. Installs Java, Spark and Apache Sedona.
2. **Sedona spatial join:** finds the district of every building and drops buildings outside Kerala. *(Module 1, 3)*
3. **Cleaning:** repairs broken shapes, removes exact duplicates and tiny shapes (< 10 m²), and flags low-confidence AI shapes. *(Module 2)*
4. Saves the clean data **split by district** (partitioned GeoParquet). *(Module 2)*
5. **Speed test:** plain Spark vs Sedona on 10k, 100k and 1M buildings. *(Module 1)*
6. **MapReduce:** district totals written as map → reduce. *(Module 1)*

**How to run:** Runtime → Run all. Allow Google Drive access. It takes about 20–40 minutes.

Everything printed is also saved to `MyDrive/rooftop_solar/logs/02_log.txt`. If the tab closes, send Claude the text of that file.

In [ ]:
# Step 1: connect Google Drive, set folders, copy the raw data to the fast Colab disk
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, time

PROJECT = '/content/drive/MyDrive/rooftop_solar'
RAW = f'{PROJECT}/data/raw'
CLEAN = f'{PROJECT}/data/clean'
RESULTS = f'{PROJECT}/results'
LOGS = f'{PROJECT}/logs'
LOCAL = '/content/work'      # fast Colab disk, deleted when the session ends
for folder in (CLEAN, RESULTS, LOGS, LOCAL):
    os.makedirs(folder, exist_ok=True)

LOG_FILE = f'{LOGS}/02_log.txt'
open(LOG_FILE, 'w').close()


def log(*parts):
    """Print, and also save to Google Drive so results survive if the tab closes."""
    text = ' '.join(str(p) for p in parts)
    print(text)
    with open(LOG_FILE, 'a') as f:
        f.write(text + '\n')


def log_table(pdf, csv_name=None):
    """Log a small pandas table; optionally save it as a CSV in results/."""
    log(pdf.to_string(index=False))
    if csv_name:
        pdf.to_csv(f'{RESULTS}/{csv_name}', index=False)


for name in ('buildings_kerala_box.parquet', 'kerala_boundaries.parquet'):
    shutil.copy(f'{RAW}/{name}', f'{LOCAL}/{name}')
log('Raw files copied to', LOCAL)

In [ ]:
# Step 2: install Java 17, PySpark 4.1 and Apache Sedona 1.9.1 (about 2-4 minutes)
import subprocess


def java_version():
    java_bin = os.path.join(os.environ['JAVA_HOME'], 'bin', 'java') if 'JAVA_HOME' in os.environ else 'java'
    out = subprocess.run(f'{java_bin} -version 2>&1', shell=True, capture_output=True, text=True).stdout
    return out.splitlines()[0] if out else 'no java'


if not any(f'version "{v}' in java_version() for v in ('17', '21')):
    !apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
    os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
!pip install -q pyspark==4.1.3 apache-sedona==1.9.1
log('Java:', java_version())

# Sedona's Java files, downloaded directly. Spark 4.1 ships its own copy of the JTS geometry library,
# so Sedona must sit on the same class path as Spark (see Step 3) instead of using spark.jars.packages.
import urllib.request

JARS = '/content/jars'
MAVEN = 'https://repo1.maven.org/maven2'
os.makedirs(JARS, exist_ok=True)
JAR_FILES = []
for url in (f'{MAVEN}/org/apache/sedona/sedona-spark-shaded-4.1_2.13/1.9.1/sedona-spark-shaded-4.1_2.13-1.9.1.jar',
            f'{MAVEN}/org/datasyslab/geotools-wrapper/1.9.1-33.5/geotools-wrapper-1.9.1-33.5.jar'):
    path = f'{JARS}/{url.rsplit("/", 1)[1]}'
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    JAR_FILES.append(path)
log('Sedona JARs:', [os.path.basename(p) for p in JAR_FILES])

In [ ]:
# Step 3: start Spark with Sedona, using all CPU cores and most of the RAM
import psutil
from sedona.spark import SedonaContext

CORES = os.cpu_count()
MEM_GB = max(2, int(psutil.virtual_memory().total / 1e9 * 0.6))   # leave room for Python and the OS

config = (
    SedonaContext.builder()
    .master('local[*]')
    .appName('rooftop_solar')
    .config('spark.driver.memory', f'{MEM_GB}g')
    .config('spark.sql.shuffle.partitions', str(CORES * 4))
    .config('spark.driver.extraClassPath', ':'.join(JAR_FILES))
    .config('spark.sql.maxConcurrentOutputFileWriters', '20')   # write districts without sorting first
    .getOrCreate()
)
sedona = SedonaContext.create(config)
log(f'Spark {sedona.version} | CPU cores: {CORES} | Spark memory: {MEM_GB} GB')

In [ ]:
# Step 4: Sedona spatial join (building -> district) + cleaning, then save split by district
from pyspark.sql import functions as F

# Buildings: the shape is stored as WKB bytes. A 64-bit fingerprint of those bytes finds exact duplicates
# without moving the shapes themselves around (moving 19 million shapes is slow on Colab's disk).
raw = sedona.read.parquet(f'{LOCAL}/buildings_kerala_box.parquet').drop('bbox')
buildings = (raw.withColumn('wkb_hash', F.xxhash64('geometry'))
                .withColumn('geometry', F.expr('ST_GeomFromWKB(geometry)')))
buildings.createOrReplaceTempView('buildings')

# Districts: each boundary has 1,250-4,000 points. Cutting it into small pieces (<= 128 points)
# makes every point-in-polygon test much faster.
districts = (sedona.read.format('geoparquet').load(f'{LOCAL}/kerala_boundaries.parquet')
             .filter("subtype = 'county'")
             .selectExpr('name AS district', 'geometry AS dgeom'))
districts.createOrReplaceTempView('districts')
district_pieces = sedona.sql('SELECT district, ST_SubDivideExplode(dgeom, 128) AS dgeom FROM districts').cache()
district_pieces.createOrReplaceTempView('district_pieces')
log(f'District boundaries cut into {district_pieces.count():,} small pieces')

BOX_TOTAL = raw.count()
log(f'Buildings in the download box: {BOX_TOTAL:,}')

# Exact duplicates: fingerprints that appear more than once, and the id of the copy we keep
hashed = buildings.select('id', 'wkb_hash')
dup_hashes = hashed.groupBy('wkb_hash').count().filter('count > 1').select('wkb_hash')
dups = (hashed.join(F.broadcast(dup_hashes), 'wkb_hash')
        .groupBy('wkb_hash').agg(F.min('id').alias('first_id'))
        .cache())
dups.createOrReplaceTempView('dups')
log(f'Shapes that appear more than once: {dups.count():,}')

t0 = time.time()
clean_df = sedona.sql("""
    WITH joined AS (      -- which district is each building in? (tested with its centre point)
        SELECT /*+ BROADCAST(d) */ b.id, b.source, b.confidence, b.class, b.wkb_hash,
               b.geometry AS geom_raw, d.district
        FROM (SELECT *, ST_Centroid(geometry) AS centre FROM buildings) b
        JOIN district_pieces d ON ST_Contains(d.dgeom, b.centre)
    ),
    checked AS (
        SELECT *, ST_IsValid(geom_raw) AS was_valid FROM joined
    ),
    fixed AS (            -- repair broken shapes, mark extra copies of duplicates
        SELECT /*+ BROADCAST(u) */ c.*,
               CASE WHEN c.was_valid THEN c.geom_raw ELSE ST_MakeValid(c.geom_raw) END AS geometry,
               coalesce(u.first_id IS NOT NULL AND c.id <> u.first_id, false) AS is_duplicate
        FROM checked c LEFT JOIN dups u ON c.wkb_hash = u.wkb_hash
    ),
    measured AS (         -- roof area in m2
        SELECT *, ST_AreaSpheroid(geometry) AS area_m2 FROM fixed
    )
    SELECT id, district, source, confidence, class, area_m2,
           NOT was_valid                                                      AS fixed_invalid,
           is_duplicate,
           area_m2 < 10                                                       AS too_small,
           coalesce(source = 'Google Open Buildings' AND confidence < 0.70, false) AS low_confidence,
           coalesce(NOT is_duplicate AND area_m2 >= 10, false)                AS keep,
           geometry
    FROM measured
""")
clean_df.write.format('geoparquet').partitionBy('district').mode('overwrite').save(f'{LOCAL}/buildings_clean')
log(f'Join + cleaning + save took {(time.time() - t0) / 60:.1f} min')

In [ ]:
# Step 5: cleaning report + district table (send these to Claude)
clean = sedona.read.format('geoparquet').load(f'{LOCAL}/buildings_clean')
clean.createOrReplaceTempView('clean')

report = sedona.sql(f"""
    SELECT {BOX_TOTAL}              AS in_download_box,
           {BOX_TOTAL} - count(*)   AS outside_kerala,
           count(*)                 AS inside_kerala,
           sum(CAST(fixed_invalid AS INT))          AS repaired_invalid,
           sum(CAST(is_duplicate AS INT))           AS duplicates_removed,
           sum(CAST(too_small AS INT))              AS too_small_removed,
           sum(CAST(keep AS INT))                   AS kept,
           sum(CAST(keep AND low_confidence AS INT)) AS kept_but_low_confidence
    FROM clean
""").toPandas()
log('\nCLEANING REPORT')
log_table(report.T.rename(columns={0: 'count'}).reset_index().rename(columns={'index': 'step'}),
          'cleaning_report.csv')

district_table = sedona.sql("""
    SELECT district,
           count(*)                                                  AS buildings,
           sum(CAST(keep AS INT))                                    AS kept,
           round(percentile_approx(CASE WHEN keep THEN area_m2 END, 0.5), 1) AS median_roof_m2,
           round(sum(CASE WHEN keep THEN area_m2 END) / 1e6, 1)      AS total_roof_km2
    FROM clean
    GROUP BY district
    ORDER BY buildings DESC
""").toPandas()
log('\nBY DISTRICT')
log_table(district_table, 'district_summary.csv')

In [ ]:
# Step 6: speed test - plain Spark (Python + Shapely, no spatial index) vs Sedona (spatial join),
# with whole district boundaries and with boundaries cut into small pieces
import pandas as pd
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
from shapely import wkb
from shapely.geometry import Point

SAMPLE_SIZES = [10_000, 100_000, 1_000_000]

# Plain Spark: send the 14 district shapes to every worker and test each building against each one
shapes = [(r.district, wkb.loads(bytes(r.wkb)))
          for r in districts.selectExpr('district', 'ST_AsBinary(dgeom) AS wkb').collect()]
shapes_bc = sedona.sparkContext.broadcast(shapes)


@udf(StringType())
def find_district(x, y):
    point = Point(x, y)
    for name, shape in shapes_bc.value:
        if shape.contains(point):
            return name
    return None


points_all = buildings.selectExpr('id', 'ST_X(ST_Centroid(geometry)) AS x', 'ST_Y(ST_Centroid(geometry)) AS y')
rows = []
for n in SAMPLE_SIZES:
    pts = (points_all.sample(fraction=min(1.0, 1.2 * n / BOX_TOTAL), seed=42).limit(n)
           .selectExpr('id', 'x', 'y', 'ST_Point(x, y) AS pt').cache())
    pts.count()                                   # load the sample into memory before timing
    pts.createOrReplaceTempView('pts')

    t = time.time()
    matched_plain = pts.withColumn('district', find_district('x', 'y')).filter('district IS NOT NULL').count()
    t_plain = time.time() - t

    t = time.time()                               # Sedona with the whole district boundaries
    sedona.sql("""
        SELECT /*+ BROADCAST(d) */ p.id FROM pts p JOIN districts d ON ST_Contains(d.dgeom, p.pt)
    """).count()
    t_whole = time.time() - t

    t = time.time()                               # Sedona with boundaries cut into small pieces
    matched_sedona = sedona.sql("""
        SELECT /*+ BROADCAST(d) */ p.id FROM pts p JOIN district_pieces d ON ST_Contains(d.dgeom, p.pt)
    """).count()
    t_sedona = time.time() - t

    rows.append({'buildings': n, 'plain_spark_s': round(t_plain, 1),
                 'sedona_whole_s': round(t_whole, 1), 'sedona_pieces_s': round(t_sedona, 1),
                 'speedup_x': round(t_plain / max(t_sedona, 0.01), 1),
                 'matched_plain': matched_plain, 'matched_sedona': matched_sedona, 'note': 'measured'})
    log(rows[-1])
    pts.unpersist()

# All buildings: Sedona only. Plain Spark would take too long, so it is estimated from the largest sample.
t = time.time()
matched_all = sedona.sql("""
    SELECT /*+ BROADCAST(d) */ b.id
    FROM (SELECT id, ST_Centroid(geometry) AS centre FROM buildings) b
    JOIN district_pieces d ON ST_Contains(d.dgeom, b.centre)
""").count()
t_all = time.time() - t
plain_estimate = rows[-1]['plain_spark_s'] * BOX_TOTAL / SAMPLE_SIZES[-1]
rows.append({'buildings': BOX_TOTAL, 'plain_spark_s': round(plain_estimate, 1),
             'sedona_whole_s': None, 'sedona_pieces_s': round(t_all, 1),
             'speedup_x': round(plain_estimate / max(t_all, 0.01), 1),
             'matched_plain': None, 'matched_sedona': matched_all,
             'note': 'plain Spark estimated; Sedona measured (incl. reading file)'})

log('\nSPARK vs SEDONA (seconds)')
log_table(pd.DataFrame(rows).astype({'matched_plain': 'Int64', 'sedona_whole_s': 'Float64'}),
          'spark_vs_sedona.csv')

In [ ]:
# Step 7: the same district totals written as MapReduce (map -> reduce) vs Spark SQL
t = time.time()
mr = (clean.filter('keep').select('district', 'area_m2').rdd
      .map(lambda r: (r.district, (1, r.area_m2)))            # map: (district, (1 building, its area))
      .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))    # reduce: add up per district
      .collect())
t_mr = time.time() - t

t = time.time()
sedona.sql('SELECT district, count(*), sum(area_m2) FROM clean WHERE keep GROUP BY district').collect()
t_sql = time.time() - t

mr_table = (pd.DataFrame([(d, c, round(a / 1e6, 2)) for d, (c, a) in mr],
                         columns=['district', 'buildings', 'roof_km2'])
            .sort_values('buildings', ascending=False))
log('\nMAPREDUCE RESULT')
log_table(mr_table)
log(f'MapReduce (RDD + Python): {t_mr:.1f} s | Spark SQL: {t_sql:.1f} s')

In [ ]:
# Step 8: copy the clean data (split by district) to Google Drive
t = time.time()
shutil.rmtree(f'{CLEAN}/buildings_clean', ignore_errors=True)    # remove an older copy first
shutil.copytree(f'{LOCAL}/buildings_clean', f'{CLEAN}/buildings_clean')
log(f'Saved {CLEAN}/buildings_clean in {time.time() - t:.0f} s')
log('ALL DONE')

In [ ]:
# Optional: open the Spark UI in a new tab and take screenshots of the Jobs and Stages pages for the report
from google.colab import output
output.serve_kernel_port_as_window(4040, path='/jobs/')

## Done

Send Claude the tables from **Step 5** (cleaning report, by district), **Step 6** (Spark vs Sedona) and **Step 7** (MapReduce), or any error.

Results are also saved as CSV files in `MyDrive/rooftop_solar/results/`.